# Full reproduction in Google Colab

Run the cell below. It downloads the benchmark repository, prepares the required Python environment, runs the full reproduction and downloads the results as a ZIP file. The first run may take a few minutes.


In [ ]:
REPOSITORY_URL = "https://github.com/myresearchbvp/ERP-MCDA-Benchmark.git"
EXACT_COMMIT = "cb3c60a3fe4038e8c51c7f65d69f60d63a8ae3dc"

from pathlib import Path
import contextlib
import csv
import io
import shutil
import subprocess
import sys
from google.colab import files

if not REPOSITORY_URL or not EXACT_COMMIT:
    raise ValueError("Repository URL and commit are required.")

base = Path.cwd()
repo = base / "erp_mcda_repository"
runtime_root = base / "erp_mcda_runtime"
package_root = base / "erp_mcda_results"

for path in (repo, runtime_root, package_root):
    if path.exists():
        shutil.rmtree(path)

def run_silent(cmd, cwd=None):
    cp = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if cp.returncode != 0:
        print(cp.stdout, end="", flush=True)
        raise RuntimeError(f"Command failed with exit code {cp.returncode}.")
    return cp.stdout.strip()

def quietly(func, *args, **kwargs):
    log = io.StringIO()
    try:
        with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            return func(*args, **kwargs)
    except Exception as exc:
        print(log.getvalue(), end="", flush=True)
        print(f"REPRODUCTION_FAILURE: {type(exc).__name__}: {exc}", flush=True)
        raise

print("Downloading the repository...", flush=True)
run_silent(["git", "clone", "--quiet", REPOSITORY_URL, str(repo)])
run_silent(["git", "checkout", EXACT_COMMIT], cwd=repo)
head = run_silent(["git", "rev-parse", "HEAD"], cwd=repo)
expected = run_silent(["git", "rev-parse", EXACT_COMMIT], cwd=repo)
if head != expected:
    raise RuntimeError(f"HEAD mismatch: expected {expected}, observed {head}")

sys.path.insert(0, str(repo / "src"))
from pipeline.colab_runtime import prepare_runtime, run_full_reproduction

print("Preparing Python 3.13.5 and the required packages...", flush=True)
python_exe, route, info = quietly(prepare_runtime, repo, runtime_root)
print(f"Environment ready: {info.replace(';', ',')}", flush=True)

reproduced = repo / "reproduced"
print("Running the full reproduction...", flush=True)
quietly(run_full_reproduction, python_exe, repo, reproduced)

def passed_rows(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return len(rows), sum(row.get("status") == "PASS" for row in rows)

parity_total, parity_pass = passed_rows(reproduced / "PARITY_REPORT.csv")
pub_total, pub_pass = passed_rows(reproduced / "PUBLICATION_PARITY.csv")

if parity_pass != parity_total or pub_pass != pub_total:
    raise RuntimeError("The reproduction completed but one or more verification checks did not pass.")

print("Full reproduction completed successfully.", flush=True)
print(f"Parity checks: {parity_pass}/{parity_total} passed", flush=True)
print(f"Publication checks: {pub_pass}/{pub_total} passed", flush=True)

shutil.copytree(
    reproduced,
    package_root,
    ignore=shutil.ignore_patterns("FULL_REPRODUCTION_PASS.txt", "COLAB_RUNTIME_INFO.txt"),
)

(package_root / "RUN_INFO.txt").write_text(
    "ERP-MCDA benchmark full reproduction\n"
    f"Repository: {REPOSITORY_URL.removesuffix('.git')}\n"
    f"Commit: {EXACT_COMMIT}\n"
    f"Runtime: {info}\n"
    f"Parity checks: {parity_pass}/{parity_total} passed\n"
    f"Publication checks: {pub_pass}/{pub_total} passed\n",
    encoding="utf-8",
)

archive_base = base / "erp_mcda_full_reproduction_result"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=package_root))
print(f"Downloading {archive_path.name}...", flush=True)
files.download(str(archive_path))
